In [4]:
import asyncio
import websockets
import json

# 수집할 코인 5개
COINS = ['btcusdt', 'ethusdt', 'bnbusdt', 'solusdt', 'xrpusdt']

# Binance WebSocket 주소
streams = '/'.join([f"{coin}@ticker" for coin in COINS])
URL = f"wss://stream.binance.com:9443/stream?streams={streams}"

async def collect_prices():
    async with websockets.connect(URL) as ws:
        print("Binance WebSocket 연결 성공!")
        for _ in range(10):  # 일단 10개만 받아보기
            msg = await ws.recv()
            data = json.loads(msg)
            symbol = data['data']['s']  # 코인 심볼
            price  = data['data']['c']  # 현재가
            print(f"{symbol}: ${price}")

await collect_prices()

Binance WebSocket 연결 성공!
SOLUSDT: $89.50000000
BTCUSDT: $70271.16000000
ETHUSDT: $2124.26000000
BNBUSDT: $641.20000000
XRPUSDT: $1.41610000
SOLUSDT: $89.50000000
BTCUSDT: $70271.17000000
ETHUSDT: $2124.33000000
BNBUSDT: $641.26000000
XRPUSDT: $1.41630000


In [5]:
from kafka import KafkaProducer
import json

# Kafka Producer 연결
producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda x: json.dumps(x).encode('utf-8')
)

print("Kafka Producer 연결 성공!")

Kafka Producer 연결 성공!


In [6]:
import psycopg2
import requests
from datetime import datetime, timezone

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="crypto",
    user="postgres",
    password="password"
)
cursor = conn.cursor()
print("DB 연결 성공!")

DB 연결 성공!


In [7]:
def fill_historical_data(days=365):
    coins = ['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'SOLUSDT', 'XRPUSDT']
    
    for coin in coins:
        print(f"{coin} 과거 데이터 수집 시작...")
        
        cursor.execute("""
            SELECT MAX(time) FROM crypto_prices 
            WHERE symbol = %s AND source = 'historical'
        """, (coin,))
        last_time = cursor.fetchone()[0]
        
        if last_time:
            start_time = int(last_time.timestamp() * 1000)
            print(f"{coin} 마지막 저장 시간: {last_time}")
        else:
            start_time = int((datetime.now(timezone.utc).timestamp() - days * 86400) * 1000)
            print(f"{coin} 처음 수집 - {days}일치 가져오기")
        
        total = 0
        while True:
            url = "https://api.binance.com/api/v3/klines"
            params = {
                'symbol': coin,
                'interval': '1m',
                'startTime': start_time,
                'limit': 1000
            }
            response = requests.get(url, params=params)
            data = response.json()
            
            if not data:
                break
            
            for candle in data:
                cursor.execute("""
                    INSERT INTO crypto_prices (time, symbol, moving_avg, min_price, max_price, source)
                    VALUES (to_timestamp(%s / 1000.0), %s, %s, %s, %s, 'historical')
                    ON CONFLICT DO NOTHING
                """, (
                    candle[6],
                    coin,
                    float(candle[4]),
                    float(candle[3]),
                    float(candle[2]),
                ))
            
            conn.commit()
            total += len(data)
            print(f"{coin} {total}개 수집 완료...")
            
            start_time = data[-1][6] + 1
            
            if start_time >= int(datetime.now(timezone.utc).timestamp() * 1000):
                break
        
        print(f"{coin} 총 {total}개 수집 완료!")

In [8]:
fill_historical_data(days=365)

BTCUSDT 과거 데이터 수집 시작...
BTCUSDT 마지막 저장 시간: 2026-03-22 10:04:59.999000+00:00
BTCUSDT 1000개 수집 완료...
BTCUSDT 1576개 수집 완료...
BTCUSDT 총 1576개 수집 완료!
ETHUSDT 과거 데이터 수집 시작...
ETHUSDT 마지막 저장 시간: 2026-03-22 10:10:59.999000+00:00
ETHUSDT 1000개 수집 완료...
ETHUSDT 1570개 수집 완료...
ETHUSDT 총 1570개 수집 완료!
BNBUSDT 과거 데이터 수집 시작...
BNBUSDT 마지막 저장 시간: 2026-03-22 10:16:59.999000+00:00
BNBUSDT 1000개 수집 완료...
BNBUSDT 1564개 수집 완료...
BNBUSDT 총 1564개 수집 완료!
SOLUSDT 과거 데이터 수집 시작...
SOLUSDT 마지막 저장 시간: 2026-03-22 10:23:59.999000+00:00
SOLUSDT 1000개 수집 완료...
SOLUSDT 1557개 수집 완료...
SOLUSDT 총 1557개 수집 완료!
XRPUSDT 과거 데이터 수집 시작...
XRPUSDT 마지막 저장 시간: 2026-03-22 10:29:59.999000+00:00
XRPUSDT 1000개 수집 완료...
XRPUSDT 1551개 수집 완료...
XRPUSDT 총 1551개 수집 완료!


In [ ]:
async def collect_and_send_loop():
    async with websockets.connect(URL) as ws:
        print("실시간 수집 시작!")
        while True:
            msg = await ws.recv()
            data = json.loads(msg)
            symbol = data['data']['s']
            price  = data['data']['c']
            
            payload = {
                'symbol': symbol,
                'price': float(price),
                'timestamp': data['data']['E']
            }
            
            producer.send('raw-prices', value=payload)
            print(f"전송 → {symbol}: ${price}")

await collect_and_send_loop()

실시간 수집 시작!
전송 → BTCUSDT: $70265.21000000
전송 → ETHUSDT: $2123.61000000
전송 → BNBUSDT: $641.35000000
전송 → XRPUSDT: $1.41580000
전송 → SOLUSDT: $89.46000000
전송 → BTCUSDT: $70265.20000000
전송 → ETHUSDT: $2123.60000000
전송 → BNBUSDT: $641.36000000
전송 → XRPUSDT: $1.41580000
전송 → SOLUSDT: $89.47000000
전송 → BTCUSDT: $70268.49000000
전송 → ETHUSDT: $2123.86000000
전송 → BNBUSDT: $641.40000000
전송 → XRPUSDT: $1.41580000
전송 → SOLUSDT: $89.51000000
전송 → BTCUSDT: $70268.50000000
전송 → ETHUSDT: $2123.86000000
전송 → BNBUSDT: $641.41000000
전송 → XRPUSDT: $1.41610000
전송 → SOLUSDT: $89.48000000
전송 → BTCUSDT: $70271.48000000
전송 → ETHUSDT: $2124.25000000
전송 → BNBUSDT: $641.45000000
전송 → XRPUSDT: $1.41610000
전송 → SOLUSDT: $89.48000000
전송 → BTCUSDT: $70271.58000000
전송 → ETHUSDT: $2124.18000000
전송 → BNBUSDT: $641.48000000
전송 → XRPUSDT: $1.41610000
전송 → SOLUSDT: $89.47000000
전송 → BTCUSDT: $70271.58000000
전송 → ETHUSDT: $2124.18000000
전송 → BNBUSDT: $641.47000000
전송 → XRPUSDT: $1.41620000
전송 → SOLUSDT: $89.47000000
전송 → BTCU